# Tensor Operations

The custom `Tensor` stores a flat row-major list plus shape and stride metadata; NumPy supplies the broadcasting/einsum comparison fixtures.

In [ ]:
from pathlib import Path
import sys
candidates = (Path.cwd() / 'code', Path.cwd().parents[1] / 'code', Path.cwd() / 'phases/01-math-foundations/12-tensor-operations/code')
code_dir = next(path for path in candidates if (path / 'tensors.py').is_file())
sys.path.insert(0, str(code_dir))
from tensors import Tensor
import numpy as np


## Build It: storage and shape transforms

A `(2,3)` tensor has row-major strides `(3,1)`. Reshape preserves the six values; `permute` changes axes and recomputes the destination layout.

In [ ]:
tensor = Tensor([[1, 2, 3], [4, 5, 6]])
reshaped = tensor.reshape((3, 2))
transposed = tensor.transpose(0, 1)
tensor.shape, tensor.strides, reshaped.to_list(), transposed.shape


## Use It: broadcasting and einsum

A `(hidden,)` bias aligns with the trailing axis of `(batch, sequence, hidden)`. In `ij,jk->ik`, the repeated `j` is contracted.

In [ ]:
hidden = np.zeros((2, 3, 4))
bias = np.array([1.0, 2.0, 3.0, 4.0])
added = hidden + bias
A = np.arange(6.0).reshape(2, 3)
B = np.arange(12.0).reshape(3, 4)
added.shape, np.einsum('ij,jk->ik', A, B).shape


## Ship It: attention shape trace

For `(B,H,T,D)=(2,4,8,16)`, `einsum('bhtd,bhsd->bhts')` creates `(2,4,8,8)` scores and the value contraction returns `(2,4,8,16)`.

In [ ]:
Q = np.zeros((2, 4, 8, 16))
K = np.zeros_like(Q)
V = np.zeros_like(Q)
scores = np.einsum('bhtd,bhsd->bhts', Q, K)
output = np.einsum('bhts,bhsd->bhtd', scores, V)
scores.shape, output.shape


## Exercise

Try `Tensor([[1, 2], [3]])` and a partial index such as `tensor[0]`; record the two explicit errors. Then change `T=8` to `T=5` in the attention trace and predict the score shape before running it.